# Notebook 07: Feature Engineering Experiments

**מטרה**: בדיקה שיטתית של פיצ'רים חדשים לשיפור מודל Link Prediction

**ניסויים**:
1. **Baseline** - המודל הקיים (24 פיצ'רים)
2. **+Resource Allocation** - הוספת RA Index
3. **+Node Features** - triangles, clustering, pagerank
4. **+Temporal Features** - career overlap, collaboration count
5. **+All Combined** - כל הפיצ'רים החדשים
6. **+XGBoost** - מודל חזק יותר

**מבוסס על**: Notebook 11 (Link Prediction) מחומר הקורס

In [ ]:
# Imports
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

sys.path.append('../src')
import link_prediction as lp
import link_features as lf
import graph_build as gb

np.random.seed(42)

## 1. טעינת נתונים והכנת גרפים

In [ ]:
# Load data
cast_edges = pd.read_csv('../data/processed/cast_edges.csv')
print(f"Total cast edges: {len(cast_edges):,}")
print(f"Actors: {cast_edges['actor_slug'].nunique():,}")
print(f"Movies: {cast_edges['movie_slug'].nunique():,}")
print(f"Year range: {cast_edges['year'].min()}-{cast_edges['year'].max()}")

In [ ]:
# Build temporal graphs (same as Notebook 06)
print("Building temporal graphs...")
G_1990_2015 = lp.build_graph_for_years(cast_edges, 1990, 2015)
G_1990_2020 = lp.build_graph_for_years(cast_edges, 1990, 2020)
G_1990_2025 = lp.build_graph_for_years(cast_edges, 1990, 2025)

print(f"\nG_1990_2015: {G_1990_2015.number_of_nodes()} nodes, {G_1990_2015.number_of_edges()} edges")
print(f"G_1990_2020: {G_1990_2020.number_of_nodes()} nodes, {G_1990_2020.number_of_edges()} edges")
print(f"G_1990_2025: {G_1990_2025.number_of_nodes()} nodes, {G_1990_2025.number_of_edges()} edges")

In [ ]:
# Get train/test pairs (same logic as Notebook 06)
print("\nExtracting train/test pairs...")

# Train: new pairs 2016-2020
train_pos = lp.get_new_pairs(G_1990_2015, G_1990_2020)
train_neg = lp.sample_negatives(G_1990_2015, len(train_pos), exclude_pairs=set(train_pos), seed=42)

# Test: new pairs 2021-2025
test_pos = lp.get_new_pairs(G_1990_2020, G_1990_2025)
test_neg = lp.sample_negatives(G_1990_2020, len(test_pos), exclude_pairs=set(test_pos), seed=42)

print(f"Train: {len(train_pos)} positives, {len(train_neg)} negatives")
print(f"Test:  {len(test_pos)} positives, {len(test_neg)} negatives")

train_pairs = train_pos + train_neg
test_pairs = test_pos + test_neg
y_train = np.array([1] * len(train_pos) + [0] * len(train_neg))
y_test = np.array([1] * len(test_pos) + [0] * len(test_neg))

## 2. פונקציות לחישוב פיצ'רים חדשים

In [ ]:
def add_resource_allocation(G, pairs):
    """Add Resource Allocation Index feature"""
    ra_map = {}
    for u, v, score in nx.resource_allocation_index(G, pairs):
        ra_map[(u, v)] = score
    return [ra_map.get(p, 0.0) for p in pairs]


def add_node_features(G, pairs):
    """Add node-level features: triangles, clustering, pagerank"""
    print("  Computing triangles...")
    triangles = nx.triangles(G)
    
    print("  Computing clustering...")
    clustering = nx.clustering(G)
    
    print("  Computing pagerank...")
    pagerank = nx.pagerank(G, max_iter=100)
    
    features = []
    for u, v in pairs:
        features.append([
            triangles.get(u, 0),
            triangles.get(v, 0),
            np.sqrt(triangles.get(u, 0) * triangles.get(v, 0)),  # geometric mean
            clustering.get(u, 0.0),
            clustering.get(v, 0.0),
            (clustering.get(u, 0.0) + clustering.get(v, 0.0)) / 2,  # average
            pagerank.get(u, 0.0),
            pagerank.get(v, 0.0),
            pagerank.get(u, 0.0) * pagerank.get(v, 0.0),  # product
        ])
    
    return np.array(features)


def add_temporal_features(G, pairs, cast_edges, current_year=2025):
    """Add temporal features: career overlap, collaboration frequency"""
    # Build actor -> years mapping
    actor_years = cast_edges.groupby('actor_slug')['year'].apply(set).to_dict()
    
    # Build collaboration count from MultiGraph
    features = []
    for u, v in pairs:
        u_years = actor_years.get(u, set())
        v_years = actor_years.get(v, set())
        
        # Career overlap
        overlap = len(u_years & v_years) if u_years and v_years else 0
        
        # Career length
        u_career = max(u_years) - min(u_years) + 1 if u_years else 0
        v_career = max(v_years) - min(v_years) + 1 if v_years else 0
        
        # Recent activity (last 3 years)
        u_recent = 1 if u_years and max(u_years) >= current_year - 3 else 0
        v_recent = 1 if v_years and max(v_years) >= current_year - 3 else 0
        
        # Collaboration count (from MultiGraph)
        collab_count = G.number_of_edges(u, v) if G.has_edge(u, v) else 0
        
        features.append([
            overlap,
            u_career,
            v_career,
            u_recent,
            v_recent,
            collab_count,
            len(u_years),  # total films
            len(v_years),
        ])
    
    return np.array(features)

## 3. Experiment 1: Baseline (24 פיצ'רים מקוריים)

In [ ]:
print("="*70)
print("EXPERIMENT 1: BASELINE (24 original features)")
print("="*70)

# Build baseline features (same as Notebook 06)
print("\nComputing baseline features...")
centralities_train = lp.compute_centralities(G_1990_2015)
svd_emb_train = lf.svd_node_embeddings(G_1990_2015, n_components=32, seed=42)

X_train_base, feature_names_base = lf.build_feature_matrix(
    G_1990_2015, train_pairs, centralities_train, svd_emb_train, 
    n2v_emb=None, years_ahead=5
)

centralities_test = lp.compute_centralities(G_1990_2020)
svd_emb_test = lf.svd_node_embeddings(G_1990_2020, n_components=32, seed=42)

X_test_base, _ = lf.build_feature_matrix(
    G_1990_2020, test_pairs, centralities_test, svd_emb_test,
    n2v_emb=None, years_ahead=5
)

print(f"Baseline features: {X_train_base.shape[1]}")

# Train models
lr_base, scaler_base = lp.train_logistic_regression(X_train_base, y_train, seed=42)
rf_base = lp.train_random_forest(X_train_base, y_train, seed=42)

# Evaluate
metrics_lr_base = lp.evaluate_model(lr_base, X_test_base, y_test, scaler_base)
metrics_rf_base = lp.evaluate_model(rf_base, X_test_base, y_test, scaler=None)

print("\n" + "="*50)
print("BASELINE RESULTS")
print("="*50)
print(f"Logistic Regression - AUC-ROC: {metrics_lr_base['auc_roc']:.4f}")
print(f"Random Forest       - AUC-ROC: {metrics_rf_base['auc_roc']:.4f}")
print("="*50)

## 4. Experiment 2: +Resource Allocation

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 2: +RESOURCE ALLOCATION (25 features)")
print("="*70)

print("\nAdding Resource Allocation Index...")
ra_train = add_resource_allocation(G_1990_2015, train_pairs)
ra_test = add_resource_allocation(G_1990_2020, test_pairs)

X_train_ra = np.column_stack([X_train_base, ra_train])
X_test_ra = np.column_stack([X_test_base, ra_test])

print(f"Features with RA: {X_train_ra.shape[1]}")

# Train
lr_ra, scaler_ra = lp.train_logistic_regression(X_train_ra, y_train, seed=42)
rf_ra = lp.train_random_forest(X_train_ra, y_train, seed=42)

# Evaluate
metrics_lr_ra = lp.evaluate_model(lr_ra, X_test_ra, y_test, scaler_ra)
metrics_rf_ra = lp.evaluate_model(rf_ra, X_test_ra, y_test, scaler=None)

print("\n" + "="*50)
print("+RA RESULTS")
print("="*50)
print(f"Logistic Regression - AUC-ROC: {metrics_lr_ra['auc_roc']:.4f} (Δ={metrics_lr_ra['auc_roc']-metrics_lr_base['auc_roc']:+.4f})")
print(f"Random Forest       - AUC-ROC: {metrics_rf_ra['auc_roc']:.4f} (Δ={metrics_rf_ra['auc_roc']-metrics_rf_base['auc_roc']:+.4f})")
print("="*50)

## 5. Experiment 3: +Node Features

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3: +NODE FEATURES (33 features)")
print("="*70)

print("\nAdding node features (triangles, clustering, pagerank)...")
node_feat_train = add_node_features(G_1990_2015, train_pairs)
node_feat_test = add_node_features(G_1990_2020, test_pairs)

X_train_node = np.column_stack([X_train_base, node_feat_train])
X_test_node = np.column_stack([X_test_base, node_feat_test])

print(f"Features with node: {X_train_node.shape[1]}")

# Train
lr_node, scaler_node = lp.train_logistic_regression(X_train_node, y_train, seed=42)
rf_node = lp.train_random_forest(X_train_node, y_train, seed=42)

# Evaluate
metrics_lr_node = lp.evaluate_model(lr_node, X_test_node, y_test, scaler_node)
metrics_rf_node = lp.evaluate_model(rf_node, X_test_node, y_test, scaler=None)

print("\n" + "="*50)
print("+NODE FEATURES RESULTS")
print("="*50)
print(f"Logistic Regression - AUC-ROC: {metrics_lr_node['auc_roc']:.4f} (Δ={metrics_lr_node['auc_roc']-metrics_lr_base['auc_roc']:+.4f})")
print(f"Random Forest       - AUC-ROC: {metrics_rf_node['auc_roc']:.4f} (Δ={metrics_rf_node['auc_roc']-metrics_rf_base['auc_roc']:+.4f})")
print("="*50)

## 6. Experiment 4: +Temporal Features

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4: +TEMPORAL FEATURES (32 features)")
print("="*70)

print("\nAdding temporal features...")
temp_feat_train = add_temporal_features(G_1990_2015, train_pairs, cast_edges, current_year=2015)
temp_feat_test = add_temporal_features(G_1990_2020, test_pairs, cast_edges, current_year=2020)

X_train_temp = np.column_stack([X_train_base, temp_feat_train])
X_test_temp = np.column_stack([X_test_base, temp_feat_test])

print(f"Features with temporal: {X_train_temp.shape[1]}")

# Train
lr_temp, scaler_temp = lp.train_logistic_regression(X_train_temp, y_train, seed=42)
rf_temp = lp.train_random_forest(X_train_temp, y_train, seed=42)

# Evaluate
metrics_lr_temp = lp.evaluate_model(lr_temp, X_test_temp, y_test, scaler_temp)
metrics_rf_temp = lp.evaluate_model(rf_temp, X_test_temp, y_test, scaler=None)

print("\n" + "="*50)
print("+TEMPORAL FEATURES RESULTS")
print("="*50)
print(f"Logistic Regression - AUC-ROC: {metrics_lr_temp['auc_roc']:.4f} (Δ={metrics_lr_temp['auc_roc']-metrics_lr_base['auc_roc']:+.4f})")
print(f"Random Forest       - AUC-ROC: {metrics_rf_temp['auc_roc']:.4f} (Δ={metrics_rf_temp['auc_roc']-metrics_rf_base['auc_roc']:+.4f})")
print("="*50)

## 7. Experiment 5: ALL COMBINED

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 5: ALL FEATURES COMBINED (42 features)")
print("="*70)

print("\nCombining all new features...")
X_train_all = np.column_stack([X_train_base, ra_train, node_feat_train, temp_feat_train])
X_test_all = np.column_stack([X_test_base, ra_test, node_feat_test, temp_feat_test])

print(f"Total features: {X_train_all.shape[1]}")

# Train
lr_all, scaler_all = lp.train_logistic_regression(X_train_all, y_train, seed=42)
rf_all = lp.train_random_forest(X_train_all, y_train, seed=42)

# Evaluate
metrics_lr_all = lp.evaluate_model(lr_all, X_test_all, y_test, scaler_all)
metrics_rf_all = lp.evaluate_model(rf_all, X_test_all, y_test, scaler=None)

print("\n" + "="*50)
print("ALL COMBINED RESULTS")
print("="*50)
print(f"Logistic Regression - AUC-ROC: {metrics_lr_all['auc_roc']:.4f} (Δ={metrics_lr_all['auc_roc']-metrics_lr_base['auc_roc']:+.4f})")
print(f"Random Forest       - AUC-ROC: {metrics_rf_all['auc_roc']:.4f} (Δ={metrics_rf_all['auc_roc']-metrics_rf_base['auc_roc']:+.4f})")
print("="*50)

## 8. Experiment 6: XGBoost on Best Features

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 6: XGBoost on ALL FEATURES")
print("="*70)

try:
    from xgboost import XGBClassifier
    
    print("\nTraining XGBoost...")
    # Scale pos_weight by class imbalance
    scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
    
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    
    xgb.fit(X_train_all, y_train)
    
    # Evaluate
    y_pred_proba_xgb = xgb.predict_proba(X_test_all)[:, 1]
    metrics_xgb = {
        'auc_roc': roc_auc_score(y_test, y_pred_proba_xgb),
        'avg_precision': average_precision_score(y_test, y_pred_proba_xgb)
    }
    
    y_pred_xgb = xgb.predict(X_test_all)
    metrics_xgb['precision'] = precision_score(y_test, y_pred_xgb)
    metrics_xgb['recall'] = recall_score(y_test, y_pred_xgb)
    metrics_xgb['f1'] = f1_score(y_test, y_pred_xgb)
    
    print("\n" + "="*50)
    print("XGBoost RESULTS")
    print("="*50)
    print(f"XGBoost - AUC-ROC: {metrics_xgb['auc_roc']:.4f} (Δ={metrics_xgb['auc_roc']-metrics_rf_base['auc_roc']:+.4f} vs baseline RF)")
    print("="*50)
    
except ImportError:
    print("\nXGBoost not installed. Skipping Experiment 6.")
    print("Install with: pip install xgboost")
    metrics_xgb = None

## 9. סיכום תוצאות

In [ ]:
# Create summary table
results = []

results.append({
    'Experiment': '1. Baseline',
    'Features': 24,
    'LR_AUC': metrics_lr_base['auc_roc'],
    'RF_AUC': metrics_rf_base['auc_roc'],
    'Best_AUC': max(metrics_lr_base['auc_roc'], metrics_rf_base['auc_roc']),
    'Improvement': 0.0
})

results.append({
    'Experiment': '2. +Resource Allocation',
    'Features': 25,
    'LR_AUC': metrics_lr_ra['auc_roc'],
    'RF_AUC': metrics_rf_ra['auc_roc'],
    'Best_AUC': max(metrics_lr_ra['auc_roc'], metrics_rf_ra['auc_roc']),
    'Improvement': max(metrics_lr_ra['auc_roc'], metrics_rf_ra['auc_roc']) - results[0]['Best_AUC']
})

results.append({
    'Experiment': '3. +Node Features',
    'Features': 33,
    'LR_AUC': metrics_lr_node['auc_roc'],
    'RF_AUC': metrics_rf_node['auc_roc'],
    'Best_AUC': max(metrics_lr_node['auc_roc'], metrics_rf_node['auc_roc']),
    'Improvement': max(metrics_lr_node['auc_roc'], metrics_rf_node['auc_roc']) - results[0]['Best_AUC']
})

results.append({
    'Experiment': '4. +Temporal Features',
    'Features': 32,
    'LR_AUC': metrics_lr_temp['auc_roc'],
    'RF_AUC': metrics_rf_temp['auc_roc'],
    'Best_AUC': max(metrics_lr_temp['auc_roc'], metrics_rf_temp['auc_roc']),
    'Improvement': max(metrics_lr_temp['auc_roc'], metrics_rf_temp['auc_roc']) - results[0]['Best_AUC']
})

results.append({
    'Experiment': '5. ALL Combined',
    'Features': 42,
    'LR_AUC': metrics_lr_all['auc_roc'],
    'RF_AUC': metrics_rf_all['auc_roc'],
    'Best_AUC': max(metrics_lr_all['auc_roc'], metrics_rf_all['auc_roc']),
    'Improvement': max(metrics_lr_all['auc_roc'], metrics_rf_all['auc_roc']) - results[0]['Best_AUC']
})

if metrics_xgb:
    results.append({
        'Experiment': '6. XGBoost (All)',
        'Features': 42,
        'LR_AUC': np.nan,
        'RF_AUC': np.nan,
        'Best_AUC': metrics_xgb['auc_roc'],
        'Improvement': metrics_xgb['auc_roc'] - results[0]['Best_AUC']
    })

df_results = pd.DataFrame(results)

print("\n" + "="*80)
print(" " * 20 + "FINAL SUMMARY - ALL EXPERIMENTS")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

# Find best
best_idx = df_results['Best_AUC'].idxmax()
best_exp = df_results.loc[best_idx]

print(f"\n🏆 BEST PERFORMER: {best_exp['Experiment']}")
print(f"   AUC-ROC: {best_exp['Best_AUC']:.4f}")
print(f"   Improvement over baseline: +{best_exp['Improvement']:.4f} ({best_exp['Improvement']/results[0]['Best_AUC']*100:.2f}%)")
print(f"   Features: {int(best_exp['Features'])}")

## 10. ויזואליזציה

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of best AUC
colors = ['gray'] + ['steelblue'] * (len(df_results) - 1)
bars = ax1.barh(df_results['Experiment'], df_results['Best_AUC'], color=colors)
bars[best_idx].set_color('gold')
ax1.set_xlabel('AUC-ROC', fontsize=12)
ax1.set_title('Best AUC-ROC per Experiment', fontsize=14, fontweight='bold')
ax1.axvline(x=results[0]['Best_AUC'], color='red', linestyle='--', alpha=0.5, label='Baseline')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Improvement over baseline
improvements = df_results['Improvement'][1:]  # skip baseline
exp_names = df_results['Experiment'][1:]
colors_imp = ['green' if x > 0 else 'red' for x in improvements]
ax2.barh(exp_names, improvements, color=colors_imp, alpha=0.7)
ax2.set_xlabel('Improvement (Δ AUC-ROC)', fontsize=12)
ax2.set_title('Improvement over Baseline', fontsize=14, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/feature_experiments_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Saved comparison plot: figures/feature_experiments_comparison.png")

## 11. שמירת תוצאות

In [ ]:
# Save results to CSV
df_results.to_csv('../data/processed/feature_experiments_results.csv', index=False)
print("\n💾 Saved results: data/processed/feature_experiments_results.csv")

# Save detailed metrics for best model
if best_exp['Experiment'] == '5. ALL Combined':
    best_metrics = {
        'model': 'RandomForest' if metrics_rf_all['auc_roc'] > metrics_lr_all['auc_roc'] else 'LogisticRegression',
        **metrics_rf_all if metrics_rf_all['auc_roc'] > metrics_lr_all['auc_roc'] else metrics_lr_all
    }
elif best_exp['Experiment'] == '6. XGBoost (All)' and metrics_xgb:
    best_metrics = {'model': 'XGBoost', **metrics_xgb}
else:
    best_metrics = None

if best_metrics:
    pd.DataFrame([best_metrics]).to_csv('../data/processed/best_model_metrics.csv', index=False)
    print("💾 Saved best model metrics: data/processed/best_model_metrics.csv")

print("\n✅ All experiments completed!")